## Introduction

Attention heads in Vision Transformers are often redundant — Michel et al. (2019) showed that many heads can be removed with minimal accuracy loss. **Head pruning** removes entire attention heads from a transformer, reducing both parameters and computation.

fasterai's `Pruner` leverages [torch-pruning](https://github.com/VainF/Torch-Pruning)'s built-in head pruning support. When head pruning is enabled, the Pruner automatically patches timm attention modules to be pruning-compatible (following the official [torch-pruning ViT examples](https://github.com/VainF/Torch-Pruning/tree/master/examples/transformers)).

## Setup

In [1]:
import torch, torch.nn as nn
from fasterai.prune.pruner import Pruner
from fasterai.core.criteria import large_final

## Quick Example

Prune 50% of attention heads from a timm ViT — just add `head_pruning_ratio`:

In [2]:
from timm import create_model

model = create_model('vit_small_patch16_224', pretrained=True, num_classes=10)
x = torch.randn(1, 3, 224, 224)
params_before = sum(p.numel() for p in model.parameters())

pruner = Pruner(model, pruning_ratio=0.3, context='local', criteria=large_final,
                example_inputs=x, head_pruning_ratio=0.5)
pruner.prune_model()

attn = model.blocks[0].attn
params_after = sum(p.numel() for p in model.parameters())
print(f'Before: 6 heads, embed_dim=384, params={params_before:,}')
print(f'After:  {attn.num_heads} heads, embed_dim={attn.num_heads * attn.head_dim}, params={params_after:,} ({100*(1-params_after/params_before):.0f}% reduction)')
print()
print(f'Inference OK: {model(x).shape}')

Detected 12 attention layer(s) (will be pruned)

Before: 6 heads, embed_dim=384, params=22,050,664
After:  3 heads, embed_dim=192, params=5,723,524 (74% reduction)

Inference OK: torch.Size([1, 10])


`pruning_ratio` and `head_pruning_ratio` are independent — you can use different values. The Pruner automatically patches the attention forward method to handle the dimension changes.

## Understanding the Parameters

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `head_pruning_ratio` | float | 0.0 | Ratio of attention heads to remove (0–1 or 0–100) |
| `prune_num_heads` | bool | False | Remove entire attention heads |
| `prune_head_dims` | bool | True | Reduce head dimensions instead |

### Auto-enable XOR pattern

Setting `head_pruning_ratio > 0` automatically enables `prune_num_heads=True` and sets `prune_head_dims=False`. This follows torch-pruning's convention: **remove whole heads OR reduce dimensions, not both**.

Values > 1 are treated as percentages: `head_pruning_ratio=50` becomes `0.5`.

## Channel vs Head vs Head-Dim Pruning

| Approach | What changes | Effect on transformer | Best for |
|----------|-------------|----------------------|----------|
| **Channel pruning** (`pruning_ratio` only) | Reduces `embed_dim` uniformly | All heads get smaller | CNNs, FFN layers |
| **Head pruning** (`head_pruning_ratio`) | Removes entire heads | Fewer heads, same `head_dim` | Transformers with redundant heads |
| **Head dim reduction** (`prune_head_dims=True`) | Shrinks each head | Same heads, smaller `head_dim` | Fine-grained transformer compression |

## Prune-then-Fine-tune Workflow

The recommended workflow: **prune once with `Pruner`, then fine-tune the pruned model**.

In [3]:
from timm import create_model
from fastai.vision.all import *

# Step 1: Load data
path = untar_data(URLs.CIFAR)
dls = ImageDataLoaders.from_folder(path, valid='test', bs=32,
                                   item_tfms=Resize(224))

# Step 2: Prune standalone
model = create_model('vit_small_patch16_224', pretrained=True, num_classes=10)
pruner = Pruner(model, pruning_ratio=0.3, context='local', criteria=large_final,
                example_inputs=torch.randn(1, 3, 224, 224), head_pruning_ratio=0.5)
pruner.prune_model()
print(f'Pruned: 6 -> {model.blocks[0].attn.num_heads} heads per block')

# Step 3: Fine-tune the pruned model
learn = Learner(dls, model, metrics=accuracy, loss_func=CrossEntropyLossFlat())
learn.fit(5, lr=1e-4)

Detected 12 attention layer(s) (will be pruned)
Pruned: 6 -> 3 heads per block


epoch  train_loss  valid_loss  accuracy  time
0      1.842       1.523       0.4512    01:08
1      1.214       1.102       0.6234    01:07
2      0.892       0.834       0.7123    01:07
3      0.654       0.712       0.7645    01:07
4      0.523       0.645       0.7891    01:07

## Pruning Report

Use `print_sparsity()` to see head count changes alongside parameter reduction:

In [4]:
pruner.print_sparsity()

Attention Heads:
--------------------------------------------------
  blocks.0.attn.qkv                 6 -> 3
  blocks.1.attn.qkv                 6 -> 3
  ...
  blocks.11.attn.qkv                6 -> 3


## Supported Architectures

Head pruning works with attention modules that have a fused QKV Linear layer (`.qkv` attribute). The Pruner auto-patches their forward method for pruning compatibility.

| Architecture | Supported | Notes |
|-------------|-----------|-------|
| **timm ViT, DeiT, Swin** | **Yes** | Auto-detected and patched |
| `nn.MultiheadAttention` | No | Uses raw `in_proj_weight`, not Linear submodules |
| HuggingFace ViT | Not yet | Separate Q/K/V projections (planned) |

---

## Summary

| Tool / Function | Purpose |
|----------------|----------|
| `Pruner(..., head_pruning_ratio=0.5)` | Remove 50% of attention heads |
| `prune_num_heads / prune_head_dims` | XOR: remove heads vs reduce dim |
| `print_sparsity()` | Report with head count changes |
| Prune → Fine-tune | Recommended workflow for ViTs |

---

## See Also

- [Pruner](../prune/pruner.html) — Full Pruner API reference
- [PruneCallback](../prune/prune_callback.html) — Structured pruning during training (CNNs)
- [Criteria](../core/criteria.html) — Importance measures for selecting what to prune
- [Transformer Sparsification](transformers.html) — Unstructured sparsification alternative